# Serving Your Portfolio Model with FastAPI

FastAPI is a modern async web framework that auto-generates OpenAPI documentation and validates requests using Pydantic. This notebook wraps a FastAPI service around **the portfolio model you exported from AIAT 114 or AIAT 122** — the same artifact Unit 1 served — and tests it in-notebook using `TestClient`, no uvicorn process required.

Because the model is fixed across the Flask lesson, this one and the Docker lesson, every difference you observe is a property of the *serving layer*, not of the model.

## 🔗 Where this fits

**Builds on:** Course 08 (AIAT 122) — Unit 5, lesson 06 "06 Flask and FastAPI Deployment" — that lesson got a prediction out of an endpoint; here the point is everything around the prediction — Pydantic rejecting a malformed request before your model ever sees it, and the OpenAPI page generated from that same schema.

**Used later in:** Course 12 (AIAT 126) — Unit 5, lesson 01 "Project Documentation and Presentation", whose deliverable checklist asks for live-demo setup instructions and screenshots of the deployed system.


## Learning Objectives

By the end of this notebook you will be able to:
1. Explain FastAPI's key advantages over Flask (auto-docs, Pydantic, async support)
2. Generate request/response schemas from `model_card.json` rather than hard-coding field names
3. Build FastAPI endpoints for `/predict`, `/health`, and `/model-info` that report the *real* deployed model
4. Test a FastAPI app in-notebook using `TestClient`
5. Decide when to choose FastAPI vs Flask

> Need a portfolio model? See [`../../PORTFOLIO_MODEL.md`](../../PORTFOLIO_MODEL.md). Without one, this notebook serves the named fallback `wdbc-baseline` and says so in its output.

## 1. Install Dependencies

In [1]:
import subprocess, sys
subprocess.run([sys.executable, "-m", "pip", "install", "httpx", "-q"], check=True)
print("httpx ready (required by TestClient)")

httpx ready (required by TestClient)



[notice] A new release of pip is available: 26.0.1 -> 26.2.1
[notice] To update, run: pip install --upgrade pip


## 2. Load the Portfolio Model

No training happens in this lesson — the model arrived finished. We load the artifact and its card, exactly as Unit 1 did.

In [2]:
# WHAT: load YOUR portfolio model - the same artifact Unit 1 served.
# WHY: one model, several serving layers. Because the artifact does not change
# between the Flask lesson, this one, and the Docker lesson, every difference you
# observe is the serving layer, not the model. See Course 11/PORTFOLIO_MODEL.md.
import json
import sys
from pathlib import Path

for _d in [Path.cwd().resolve(), *Path.cwd().resolve().parents]:
    if (_d / "portfolio_model.py").exists():
        COURSE11 = _d
        break
else:
    raise FileNotFoundError("Could not find portfolio_model.py - run this notebook from inside Course 11.")

if str(COURSE11) not in sys.path:
    sys.path.insert(0, str(COURSE11))

import portfolio_model as pf

# Falls back to the named 'wdbc-baseline' model if you have not exported one yet.
model, card = pf.load_portfolio_model()
MODEL_DIR = pf.portfolio_dir()


FALLBACK MODEL 'wdbc-baseline' — this is NOT your model.
  directory   : /Users/abdullah/ai-diploma-portfolio
  artifact    : model.joblib  (sklearn)
  task        : classification  ->  2 classes ['malignant', 'benign']
  features    : 30 (first three: ['mean radius', 'mean texture', 'mean perimeter'])
  accuracy    : 0.9825 on held-out 20% (random_state=42, stratified)
  Export your own model from AIAT 114 or AIAT 122 and re-run: see Course 11/PORTFOLIO_MODEL.md


## 3. Pydantic Schemas — Generated From the Model Card

Pydantic models describe the **shape** of request and response data. FastAPI uses them to:
- Automatically validate incoming JSON (wrong type or missing field → 422 Unprocessable Entity)
- Generate the OpenAPI schema that powers `/docs`
- Serialize response objects to JSON

The catch: typing the field names by hand ties the API to one model forever. We build the schema at runtime with `pydantic.create_model`, straight from the card's `feature_names`.

> **A note on range constraints.** A tutorial iris API can write `Field(..., ge=0)` because petals cannot be negative. Your model's features may be standardized, signed PCA components, or engineered ratios — for most real feature sets `ge=0` would reject valid requests. So the generated schema validates **types and presence**, which are always true, and leaves range expectations documented in the card. A constraint you cannot justify for every deployment is a bug waiting to happen.

In [3]:
# WHAT: build the request schema from the card instead of typing field names.
# WHY: create_model() makes a real Pydantic class at runtime - one required float
# per feature, in the trained order, with the original column name as its
# description. Retrain with a different feature set and the schema follows.
from pydantic import BaseModel, Field, create_model
from typing import Dict

# schema_fields() maps each training column to a legal JSON field name:
# 'mean radius' -> 'mean_radius'. It refuses to continue if two columns collide.
FIELDS = pf.schema_fields(card)
FIELD_ORDER = [field for field, _ in FIELDS]

PredictRequest = create_model(
    "PredictRequest",
    **{field: (float, Field(..., description=f"training column: {original}"))
       for field, original in FIELDS},
)

class PredictionResponse(BaseModel):
    prediction: str
    class_id: int
    confidence: float
    probabilities: Dict[str, float]

print("Generated request schema with", len(FIELD_ORDER), "fields")
for field, original in FIELDS[:4]:
    print(f"  {field:<28} <- {original!r}")
print(f"  ... and {len(FIELDS) - 4} more")

# Parse the card's real sample row through the generated schema.
sample = PredictRequest(**dict(zip(FIELD_ORDER, card["sample_input"])))
print("\nSample row validated OK ->", list(sample.model_dump().items())[:2], "...")


Generated request schema with 30 fields
  mean_radius                  <- 'mean radius'
  mean_texture                 <- 'mean texture'
  mean_perimeter               <- 'mean perimeter'
  mean_area                    <- 'mean area'
  ... and 26 more

Sample row validated OK -> [('mean_radius', 11.26), ('mean_texture', 19.96)] ...


## 4. Write the FastAPI Application

Notice `async def` on each endpoint. FastAPI runs on an async event loop (ASGI). Async endpoints shine when the handler `await`s on I/O — a database, a feature store, another HTTP service — because one worker can serve other requests while waiting.

**Important catch for ML serving:** a *blocking* call like sklearn's `predict()` inside an `async def` blocks the whole event loop, so it does **not** buy you concurrency and can even hurt it. FastAPI sidesteps this automatically if you declare the endpoint as a plain `def` — it runs that in a threadpool. So for CPU-bound inference, prefer a plain `def`, or offload the heavy call (e.g. `run_in_executor`). We use `async def` here to show the syntax; just don't assume `async` alone makes blocking work concurrent.

The `/model-info` endpoint is new and it matters more than it looks: in an incident, the first question is never "is the box up?" but "**which model is up?**". Because it reads the card, it can answer that — including the embarrassing case where production is still serving the course fallback.

In [4]:
%%writefile /tmp/fastapi_app.py
# WHAT: the full FastAPI service - loader, generated schema, three endpoints.
# WHY: compare it line-for-line with the Flask version: the validation code
# disappeared into the schema, and /docs + OpenAPI come for free. Note that the
# app imports NO course helper - a deployed service depends only on the artifact
# and its card, exactly as in Unit 1 (and exactly as it must inside a container).
import json
import os
import re
from pathlib import Path

import numpy as np
from fastapi import FastAPI
from pydantic import BaseModel, Field, create_model
from typing import Dict

# --- Artifact + card --------------------------------------------------------
MODEL_DIR = Path(os.environ.get("AI_DIPLOMA_PORTFOLIO",
                                str(Path.home() / "ai-diploma-portfolio")))
CARD = json.loads((MODEL_DIR / "model_card.json").read_text())
ARTIFACT = MODEL_DIR / CARD["artifact"]

if CARD["framework"] == "sklearn":
    import joblib
    _model = joblib.load(ARTIFACT)

    def probabilities(rows):
        return np.asarray(_model.predict_proba(rows))

elif CARD["framework"] == "onnx":
    import onnxruntime as ort
    _session = ort.InferenceSession(str(ARTIFACT), providers=["CPUExecutionProvider"])
    _input = _session.get_inputs()[0].name

    def probabilities(rows):
        logits = np.asarray(_session.run(None, {_input: np.asarray(rows, dtype=np.float32)})[0])
        exp = np.exp(logits - logits.max(axis=1, keepdims=True))
        return exp / exp.sum(axis=1, keepdims=True)

else:
    raise RuntimeError(f"This app serves 'sklearn' or 'onnx' artifacts, not {CARD['framework']!r}.")

# --- Schema generated from the card ----------------------------------------
def api_field(name):
    slug = re.sub(r"\W+", "_", str(name).strip().lower()).strip("_")
    return f"f_{slug}" if not slug or slug[0].isdigit() else slug

FIELD_ORDER = [api_field(n) for n in CARD["feature_names"]]

PredictRequest = create_model(
    "PredictRequest",
    **{field: (float, Field(..., description=f"training column: {original}"))
       for field, original in zip(FIELD_ORDER, CARD["feature_names"])},
)

class PredictionResponse(BaseModel):
    prediction: str
    class_id: int
    confidence: float
    probabilities: Dict[str, float]

# --- Endpoints --------------------------------------------------------------
app = FastAPI(
    title=f"{CARD['name']} API",
    description=f"Serving the AIAT 125 portfolio model from {CARD['source_course']}",
    version="1.0.0",
)

@app.get("/health")
async def health():
    return {"status": "ok", "model": CARD["name"]}

@app.get("/model-info")
async def model_info():
    """Answer the question every on-call engineer asks first: what is actually deployed?"""
    return {
        "name": CARD["name"],
        "source_course": CARD["source_course"],
        "framework": CARD["framework"],
        "task": CARD["task"],
        "n_features": len(CARD["feature_names"]),
        "classes": CARD["class_names"],
        "reported_metric": CARD["metric"],
        "is_fallback": bool(CARD.get("is_fallback", False)),
    }

@app.post("/predict", response_model=PredictionResponse)
async def predict(req: PredictRequest):
    row = [getattr(req, field) for field in FIELD_ORDER]
    proba = probabilities([row])[0]
    idx = int(np.argmax(proba))
    return PredictionResponse(
        prediction=CARD["class_names"][idx],
        class_id=idx,
        confidence=round(float(proba[idx]), 4),
        probabilities={name: round(float(p), 4)
                       for name, p in zip(CARD["class_names"], proba)},
    )


Overwriting /tmp/fastapi_app.py


## 5. Test with TestClient

`TestClient` wraps the ASGI app and lets you make HTTP calls synchronously in a notebook — no uvicorn server needed. It uses `httpx` internally.

In [5]:
# WHAT: exercise every endpoint through FastAPI's in-process TestClient.
# WHY: note where the work went - malformed input returns 422 from Pydantic
# WITHOUT a single line of validation code of ours running.
import importlib
import sys

sys.path.insert(0, "/tmp")
import fastapi_app as fmod
importlib.reload(fmod)

# TestClient calls the app in-process: no server, no port, same code paths.
from fastapi.testclient import TestClient
client = TestClient(fmod.app)

print("Health     :", client.get("/health").status_code, client.get("/health").json())

info = client.get("/model-info").json()
print("Model info :", json.dumps(info, indent=2))

# Valid prediction, built from the card's real sample row.
payload = dict(zip(fmod.FIELD_ORDER, card["sample_input"]))
resp = client.post("/predict", json=payload)
print("\nPredict    :", resp.status_code, json.dumps(resp.json(), indent=2))

# Missing field -> Pydantic raises 422 automatically.
partial = {k: v for k, v in list(payload.items())[:1]}
print("\nMissing fields ->", client.post("/predict", json=partial).status_code)

# Wrong type -> Pydantic catches it.
bad = {**payload, fmod.FIELD_ORDER[0]: "big"}
print("Wrong type     ->", client.post("/predict", json=bad).status_code)


Health     : 200 {'status': 'ok', 'model': 'wdbc-baseline'}
Model info : {
  "name": "wdbc-baseline",
  "source_course": "AIAT 125 fallback",
  "framework": "sklearn",
  "task": "classification",
  "n_features": 30,
  "classes": [
    "malignant",
    "benign"
  ],
  "reported_metric": {
    "name": "accuracy",
    "value": 0.9825,
    "split": "held-out 20% (random_state=42, stratified)"
  },
  "is_fallback": true
}

Predict    : 200 {
  "prediction": "benign",
  "class_id": 1,
  "confidence": 0.9991,
  "probabilities": {
    "malignant": 0.0009,
    "benign": 0.9991
  }
}

Missing fields -> 422
Wrong type     -> 422


/Users/abdullah/Downloads/AI Diploma/.venv/lib/python3.14/site-packages/fastapi/testclient.py:1: StarletteDeprecationWarning: Using `httpx` with `starlette.testclient` is deprecated; install `httpx2` instead.
  from starlette.testclient import TestClient as TestClient  # noqa


## 6. The Auto-Generated `/docs` Endpoint

When you run `uvicorn fastapi_app:app`, visiting `http://localhost:8000/docs` shows an interactive Swagger UI. You can try requests directly in the browser, see expected inputs/outputs, and download the OpenAPI spec from `/openapi.json`. This is generated automatically from your Pydantic models — no extra work.

In [6]:
# WHAT: fetch the OpenAPI schema FastAPI generated from our runtime-built model.
# WHY: this machine-readable contract powers /docs and lets client teams generate
# typed SDKs - and because it was generated from the card, it names YOUR features.
# Documentation that cannot drift out of date, because nobody typed it.
resp = client.get("/openapi.json")
schema = resp.json()

print("API title:", schema["info"]["title"])
print("Version  :", schema["info"]["version"])
print("Endpoints:", list(schema["paths"].keys()))

request_schema = schema["components"]["schemas"]["PredictRequest"]
properties = request_schema["properties"]
print(f"\nPredictRequest: {len(properties)} required fields, generated from model_card.json")
for name in list(properties)[:3]:
    print(f"  {name:<28} {properties[name].get('description', '')}")
print(f"  ... and {len(properties) - 3} more")


API title: wdbc-baseline API
Version  : 1.0.0
Endpoints: ['/health', '/model-info', '/predict']

PredictRequest: 30 required fields, generated from model_card.json
  mean_radius                  training column: mean radius
  mean_texture                 training column: mean texture
  mean_perimeter               training column: mean perimeter
  ... and 27 more


## 7. Flask vs FastAPI Decision Table

| Scenario | Choose |
|---|---|
| Team already uses Flask, simple API | Flask |
| Need auto-generated docs for a frontend team | FastAPI |
| High concurrent I/O-bound traffic | FastAPI (async) |
| Integrating into an existing Django codebase | Flask |
| New greenfield ML microservice | FastAPI |
| Strict request/response type safety | FastAPI (Pydantic) |

## 8. Summary

In this notebook you:
- Loaded your **portfolio artifact plus its card** — no training, the deployment job starts after training ends
- Generated `PredictRequest` from `model_card.json` with `create_model`, so the API contract follows the model
- Built three async FastAPI endpoints: `/health`, `/model-info`, `/predict`
- Tested the app in-notebook using `TestClient` — no uvicorn process needed
- Inspected the auto-generated OpenAPI schema from `/openapi.json` and saw your own feature names in it

FastAPI's key insight: **write Python type hints once, get validation + serialization + documentation for free.** The deployment insight stacked on top: **derive those type hints from the artifact**, and the documentation can never describe a model you are not actually serving.

**Next:** Unit 4 takes these same two files — artifact and card — and copies them into a Docker image.

## Self-Check (answer before scrolling up)

1. **What does Pydantic do when a required field is missing from the request body?** What HTTP status code does FastAPI return automatically?
2. **Why does this notebook refuse to put `ge=0` on the generated fields?** Name a feature type that would break under it.
3. **What is the `/docs` endpoint and how does it help during development?** Name one thing you can do there that you can't do with just curl.
4. **`/model-info` reports `is_fallback`. Why would a production service want to expose that?**
5. **When would you choose Flask instead of FastAPI?** Give a concrete real-world scenario.

## 📚 References

1. Fielding, R. T. (2000). *Architectural Styles and the Design of Network-based Software Architectures* (Ch. 5: REST). PhD dissertation, University of California, Irvine.
2. Olston, C., Fiedel, N., Gorovoy, K., et al. (2017). *TensorFlow-Serving: Flexible, High-Performance ML Serving*. NeurIPS Workshop on ML Systems. <https://arxiv.org/abs/1712.06139>
3. Shankar, S., Garcia, R., Hellerstein, J. M., & Parameswaran, A. G. (2022). *Operationalizing Machine Learning: An Interview Study*. arXiv. <https://arxiv.org/abs/2209.09125>
